In [ ]:
import pandas as pd
import numpy as np
import joblib
import json

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
RANDOM_STATE = 42
ARTIFACT_DIR = "artifacts/"

In [ ]:
df = pd.read_csv("/content/batsman_match_final_stage2.csv")
df["date"] = pd.to_datetime(df["date"])

feature_pipeline = joblib.load(
    "/content/Batsman_Feature_Pipeline (1).pkl"
)

print("STAGE 1 ✅ Dataset & feature pipeline loaded")

STAGE 1 ✅ Dataset & feature pipeline loaded


In [ ]:
!pip install category_encoders

In [ ]:
TARGET = "runs"

DROP_COLS = ["runs", "batsman", "matchid", "date"]

df["season"] = df["season"].astype(str).str[:4].astype(int)

X = df.drop(columns=DROP_COLS)
y = df[TARGET]

X_train = X[df["season"] <= 2020]
X_test  = X[df["season"] >= 2021]

y_train = y[df["season"] <= 2020]
y_test  = y[df["season"] >= 2021]

print("STAGE 2 ✅ Train/Test split complete")
print("➡ Train samples:", X_train.shape[0])
print("➡ Test samples:", X_test.shape[0])

STAGE 2 ✅ Train/Test split complete
➡ Train samples: 11420
➡ Test samples: 260


In [ ]:
X_train_t = feature_pipeline.transform(X_train)
X_test_t  = feature_pipeline.transform(X_test)

print("STAGE 3 ✅ Feature transformation complete")
print("➡ Train shape:", X_train_t.shape)
print("➡ Test shape:", X_test_t.shape)

STAGE 3 ✅ Feature transformation complete
➡ Train shape: (11420, 14)
➡ Test shape: (260, 14)


In [ ]:
import os

baseline_pred = X_test["avg_runs_last_10"]

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_mse = mean_squared_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(baseline_mse)

baseline_metrics = {
    "baseline_mae": round(baseline_mae, 3),
    "baseline_rmse": round(baseline_rmse, 3)
}

# Create the artifacts directory if it doesn't exist
os.makedirs(ARTIFACT_DIR, exist_ok=True)

with open(ARTIFACT_DIR + "batsman_baseline_metrics.json", "w") as f:
    json.dump(baseline_metrics, f, indent=4)

print("STAGE 4 ✅ Baseline evaluation complete")
print(baseline_metrics)

STAGE 4 ✅ Baseline evaluation complete
{'baseline_mae': 15.684, 'baseline_rmse': np.float64(21.541)}


In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=10,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_model.fit(X_train_t, y_train)

print("STAGE 5 ✅ Random Forest trained")

STAGE 5 ✅ Random Forest trained


In [ ]:
rf_pred = rf_model.predict(X_test_t)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_mse = mean_squared_error(y_test, rf_pred)
rf_rmse = np.sqrt(rf_mse)
rf_r2 = r2_score(y_test, rf_pred)

print("STAGE 6 ✅ Random Forest evaluation complete")
print("➡ MAE :", round(rf_mae, 3))
print("➡ RMSE:", round(rf_rmse, 3))
print("➡ R²  :", round(rf_r2, 3))

STAGE 6 ✅ Random Forest evaluation complete
➡ MAE : 15.509
➡ RMSE: 20.786
➡ R²  : 0.116


In [ ]:
from sklearn.metrics import mean_absolute_error

baseline_pred = X_test["avg_runs_last_10"]
baseline_mae = mean_absolute_error(y_test, baseline_pred)

print("Baseline MAE:", round(baseline_mae, 2))

Baseline MAE: 15.68


In [ ]:
from sklearn.linear_model import Ridge, ElasticNet
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

models = {
    "Baseline (avg_last_10)": None,

    "RandomForest": RandomForestRegressor(
        n_estimators=600,
        max_depth=16,
        min_samples_leaf=18,
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42
    ),


}

In [ ]:
results = []

# Baseline
baseline_pred = X_test["avg_runs_last_10"]
baseline_mae = mean_absolute_error(y_test, baseline_pred)

results.append(("Baseline (avg_last_10)", baseline_mae))

# ML models
for name, model in models.items():
    if model is None:
        continue

    model.fit(X_train_t, y_train)
    preds = model.predict(X_test_t)

    mae = mean_absolute_error(y_test, preds)
    results.append((name, mae))

In [ ]:
results_df = pd.DataFrame(
    results,
    columns=["Model", "MAE"]
).sort_values("MAE")

print(results_df)

                    Model        MAE
1            RandomForest  15.493876
0  Baseline (avg_last_10)  15.684064
2                 XGBoost  15.930707


In [ ]:
best_model, best_mae = results_df.iloc[0]

print("Best model:", best_model)
print("Best MAE:", round(best_mae, 2))
print("Baseline MAE:", round(baseline_mae, 2))

if best_mae < baseline_mae - 0.3:
    print("✅ Meaningful improvement → proceed with this model")
else:
    print("⚠️ Marginal improvement → feature engineering needed")

Best model: RandomForest
Best MAE: 15.49
Baseline MAE: 15.68
⚠️ Marginal improvement → feature engineering needed


In [ ]:
joblib.dump(rf, "batsman_rf_model.pkl")
print("ℹ️ Random Forest saved model")


ℹ️ Random Forest saved model
